# Chapter 36: FastAPI Fundamentals — Colab Notebook

This notebook actually runs the FastAPI code that the lesson page (`ch36-fastapi-fundamentals.html`) shows as reference-only. FastAPI needs a real ASGI runtime to execute, which Pyodide (the in-browser Python powering the rest of the course) cannot provide — no real network sockets in a WebAssembly sandbox.

Every cell below uses `fastapi.testclient.TestClient`, which drives the app **in-process** (no real socket, no port) — deterministic and reproducible every time you run this notebook, in Colab or anywhere else.

Run the cells top to bottom. Install dependencies first if needed:

```
!pip install -q fastapi uvicorn httpx
```


## 36.1 — The Smallest Possible FastAPI App

In [1]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="Candidate Scoring API")

@app.get("/health")
def health():
    return {"status": "ok"}

client = TestClient(app)
response = client.get("/health")
print("status:", response.status_code)
print("body:", response.json())


status: 200
body: {'status': 'ok'}


## 36.2 — A Typed Path Parameter

In [2]:
CANDIDATES = {
    1: {"name": "Alice", "score": 0.92},
    2: {"name": "Bob",   "score": 0.75},
    3: {"name": "Carol", "score": 0.81},
    4: {"name": "Dave",  "score": 0.60},
    5: {"name": "Eve",   "score": 0.55},
}

@app.get("/candidates/{candidate_id}")
def get_candidate(candidate_id: int):
    return CANDIDATES[candidate_id]

client = TestClient(app)

response = client.get("/candidates/1")
print("GET /candidates/1 ->", response.status_code, response.json())

response = client.get("/candidates/abc")
print("GET /candidates/abc ->", response.status_code)
print(response.json())


GET /candidates/1 -> 200 {'name': 'Alice', 'score': 0.92}
GET /candidates/abc -> 422
{'detail': [{'type': 'int_parsing', 'loc': ['path', 'candidate_id'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'abc'}]}


## 36.3 — Query Parameters

In [3]:
from typing import Optional

@app.get("/candidates")
def list_candidates(min_score: float = 0.0, name: Optional[str] = None):
    results = [c for c in CANDIDATES.values() if c['score'] >= min_score]
    if name:
        results = [c for c in results if c['name'].lower() == name.lower()]
    return results

client = TestClient(app)

print("all:", client.get("/candidates").json())
print("min_score=0.8:", client.get("/candidates", params={"min_score": 0.8}).json())
print("min_score=0.8&name=Alice:", client.get("/candidates", params={"min_score": 0.8, "name": "Alice"}).json())


all: [{'name': 'Alice', 'score': 0.92}, {'name': 'Bob', 'score': 0.75}, {'name': 'Carol', 'score': 0.81}, {'name': 'Dave', 'score': 0.6}, {'name': 'Eve', 'score': 0.55}]
min_score=0.8: [{'name': 'Alice', 'score': 0.92}, {'name': 'Carol', 'score': 0.81}]
min_score=0.8&name=Alice: [{'name': 'Alice', 'score': 0.92}]


## 36.4 — Pydantic Request Bodies

In [4]:
from pydantic import BaseModel, Field, ValidationError

class CandidateIn(BaseModel):
    name: str
    score: float = Field(gt=0, le=1)

good = CandidateIn(name="Frank", score=0.7)
print("valid instance:", good)

try:
    CandidateIn(name="Bad Score", score=5)
except ValidationError as e:
    print("ValidationError raised as expected:")
    print(e)


valid instance: name='Frank' score=0.7
ValidationError raised as expected:
1 validation error for CandidateIn
score
  Input should be less than or equal to 1 [type=less_than_equal, input_value=5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## 36.5 — Response Models &amp; Status Codes

In [5]:
class CandidateOut(BaseModel):
    id: int
    name: str
    score: float

@app.post("/candidates", response_model=CandidateOut, status_code=201)
def create_candidate(candidate: CandidateIn):
    new_id = max(CANDIDATES) + 1
    CANDIDATES[new_id] = candidate.model_dump()
    return {"id": new_id, **candidate.model_dump()}

client = TestClient(app)
response = client.post("/candidates", json={"name": "Frank", "score": 0.70})
print("status:", response.status_code)
print("body:", response.json())


status: 201
body: {'id': 6, 'name': 'Frank', 'score': 0.7}


## 36.6 — A Real Validation Error Response

In [6]:
client = TestClient(app)
response = client.post("/candidates", json={"name": "Bad Score", "score": 5})
print("status:", response.status_code)
print("body:")
print(response.json())


status: 422
body:
{'detail': [{'type': 'less_than_equal', 'loc': ['body', 'score'], 'msg': 'Input should be less than or equal to 1', 'input': 5, 'ctx': {'le': 1.0}}]}


## 36.7 — Testing With TestClient, End to End

In [7]:
client = TestClient(app)

response = client.get("/candidates/1")
assert response.status_code == 200
assert response.json()["name"] == "Alice"

response = client.post("/candidates", json={"name": "Grace", "score": 0.88})
assert response.status_code == 201
assert response.json()["name"] == "Grace"

response = client.post("/candidates", json={"name": "Bad", "score": 5})
assert response.status_code == 422

print("All TestClient assertions passed.")


All TestClient assertions passed.


## Mini Project: Candidate Scoring API

A fresh, self-contained app combining every piece from this chapter: get-by-id, filter-by-min-score, and a validated POST — proven with TestClient.

In [8]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import Optional

project_app = FastAPI(title="Candidate Scoring API — Mini Project")

PROJECT_CANDIDATES = {
    1: {"name": "Alice", "score": 0.92},
    2: {"name": "Bob",   "score": 0.75},
    3: {"name": "Carol", "score": 0.81},
    4: {"name": "Dave",  "score": 0.60},
    5: {"name": "Eve",   "score": 0.55},
}

class ProjectCandidateIn(BaseModel):
    name: str
    score: float = Field(gt=0, le=1)

class ProjectCandidateOut(BaseModel):
    id: int
    name: str
    score: float

@project_app.get("/candidates/{candidate_id}", response_model=ProjectCandidateOut)
def project_get_candidate(candidate_id: int):
    if candidate_id not in PROJECT_CANDIDATES:
        raise HTTPException(status_code=404, detail="Candidate not found")
    return {"id": candidate_id, **PROJECT_CANDIDATES[candidate_id]}

@project_app.get("/candidates")
def project_list_candidates(min_score: float = 0.0):
    return [
        {"id": cid, **c} for cid, c in PROJECT_CANDIDATES.items()
        if c["score"] >= min_score
    ]

@project_app.post("/candidates", response_model=ProjectCandidateOut, status_code=201)
def project_create_candidate(candidate: ProjectCandidateIn):
    new_id = max(PROJECT_CANDIDATES) + 1
    PROJECT_CANDIDATES[new_id] = candidate.model_dump()
    return {"id": new_id, **candidate.model_dump()}

client = TestClient(project_app)

r1 = client.get("/candidates/1")
assert r1.status_code == 200 and r1.json()["name"] == "Alice"

r2 = client.get("/candidates/999")
assert r2.status_code == 404

r3 = client.get("/candidates", params={"min_score": 0.8})
assert r3.status_code == 200
assert {c["name"] for c in r3.json()} == {"Alice", "Carol"}

r4 = client.post("/candidates", json={"name": "Grace", "score": 0.88})
assert r4.status_code == 201

r5 = client.post("/candidates", json={"name": "Bad", "score": 5})
assert r5.status_code == 422

print("Project checklist: PASSED (get-by-id, 404, min_score filter, validated POST, 422 on bad input)")


Project checklist: PASSED (get-by-id, 404, min_score filter, validated POST, 422 on bad input)


### Next: Chapter 37 — Dependency Injection &amp; Error Handling (Colab)